# Mantenimiento Predictivo en Planta Industrial
## Clasificación de Fallas de Equipos mediante Machine Learning

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI Machine Learning Repository)  
**Referencia:** Matzka, S. (2020). *Explainable Artificial Intelligence for Predictive Maintenance Applications*. UCI ML Repository. https://doi.org/10.24432/C5HS5C  
**Autora:** Wendy J. Hernández  
**Perfil:** Ingeniería Química · Especialización Ambiental · Data Analytics  
**GitHub:** github.com/wjhernandez

---

## Contexto de Negocio

En la industria de manufactura de proceso continuo (papel, pulpa, química, alimentos), los equipos rotativos como bombas, compresores y motores representan activos críticos cuya falla no planificada genera:

- **Paros no programados** que interrumpen la cadena de producción
- **Costos de mantenimiento correctivo** hasta 3-5 veces superiores al mantenimiento preventivo (Mobley, R.K., *An Introduction to Predictive Maintenance*, 2002)
- **Riesgos de seguridad y ambientales** asociados a fallas catastróficas

Bajo la filosofía **TPM (Total Productive Maintenance)**, el pilar de Mantenimiento Planificado busca transitar del mantenimiento reactivo al predictivo. Este proyecto demuestra cómo los datos de sensores de proceso permiten anticipar fallas con suficiente antelación para intervenir con precisión.

### Pregunta de negocio
> *¿Es posible predecir la falla de un equipo industrial con base en variables de proceso medibles en tiempo real, y clasificar el tipo de falla para orientar la acción de mantenimiento?*

### KPIs objetivo
- Reducir falsos negativos (fallas no detectadas): priorizar **Recall** sobre Precision
- Maximizar **PR-AUC** dado el desbalance de clases (~3.4% fallas)
- Calcular **MTBF** (Mean Time Between Failures) y **disponibilidad operacional** por tipo de producto

---
## 0. Configuración del Entorno

In [ ]:
# Instalación de dependencias (ejecutar solo la primera vez)
# !pip install ucimlrepo imbalanced-learn xgboost shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score, RocCurveDisplay
)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

# Carga del dataset directamente desde UCI
from ucimlrepo import fetch_ucirepo

# Configuración de visualización
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

PALETTE_INDUSTRIAL = ['#1a3a4a', '#2d7d9a', '#52b788', '#f4a261', '#e63946']
COLOR_OK = '#52b788'
COLOR_FAIL = '#e63946'

print('Entorno configurado correctamente.')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

---
## 1. Carga y Descripción del Dataset

In [ ]:
# Carga desde UCI ML Repository (requiere conexión a internet)
dataset = fetch_ucirepo(id=601)

X_raw = dataset.data.features
y_raw = dataset.data.targets

# Unir features y targets en un único DataFrame de trabajo
df = pd.concat([X_raw, y_raw], axis=1)

print('=== DIMENSIONES DEL DATASET ===')
print(f'Registros: {df.shape[0]:,}   |   Variables: {df.shape[1]}')
print()
print('=== PRIMERAS 5 FILAS ===')
df.head()

In [ ]:
print('=== DICCIONARIO DE VARIABLES ===')
variable_dict = {
    'UDI':                  'Identificador único secuencial (1–10000)',
    'Product ID':           'ID del producto: L (Low), M (Medium), H (High quality)',
    'Type':                 'Categoría de calidad del producto',
    'Air temperature [K]':  'Temperatura del aire (Kelvin) — generada con walk aleatorio alrededor de 300 K',
    'Process temperature [K]': 'Temperatura de proceso (K) — correlacionada con T_aire + 10 K',
    'Rotational speed [rpm]': 'Velocidad rotacional (rpm) — calculada desde potencia de 2860 W',
    'Torque [Nm]':          'Torque aplicado (Nm) — distribución normal, media 40 Nm, sin valores negativos',
    'Tool wear [min]':      'Desgaste acumulado de herramienta (minutos) — +2/4/5 min por tipo H/M/L',
    'Machine failure':      'TARGET PRINCIPAL: 1 si hay falla, 0 si operación normal',
    'TWF':                  'Tool Wear Failure — falla por desgaste entre 200-240 min (probabilidad 50%)',
    'HDF':                  'Heat Dissipation Failure — delta T < 8.6 K y velocidad < 1380 rpm',
    'PWF':                  'Power Failure — potencia fuera del rango [3500, 9000] W',
    'OSF':                  'Overstrain Failure — desgaste × torque excede umbral por tipo de producto',
    'RNF':                  'Random Failure — probabilidad aleatoria 0.1% independiente de proceso'
}

for var, desc in variable_dict.items():
    print(f'  {var:<30} {desc}')

In [ ]:
print('=== TIPOS DE DATOS Y VALORES NULOS ===')
info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'Únicos': df.nunique()
})
print(info_df)
print()
print('=== ESTADÍSTICAS DESCRIPTIVAS ===')
df.describe().round(2)

---
## 2. Análisis Exploratorio de Datos (EDA)

El EDA en el contexto de mantenimiento predictivo responde tres preguntas fundamentales:
1. ¿Cuál es la distribución y prevalencia de fallas? (balance de clases)
2. ¿Qué variables de proceso diferencian operación normal de falla?
3. ¿Existen patrones específicos por modo de falla o tipo de producto?

In [ ]:
# --- 2.1 Distribución de la variable target ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conteo de clases
failure_counts = df['Machine failure'].value_counts()
bars = axes[0].bar(
    ['Operación Normal', 'Falla'],
    failure_counts.values,
    color=[COLOR_OK, COLOR_FAIL],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar, count in zip(bars, failure_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count:,}\n({count/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Balance de Clases — Variable Target', fontsize=13, fontweight='bold', pad=15)
axes[0].set_ylabel('Número de registros')
axes[0].set_ylim(0, 11000)

# Distribución por tipo de falla
failure_types = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_labels = ['Desgaste\n(TWF)', 'Disipación\nCalor (HDF)', 'Potencia\n(PWF)', 'Sobresfuerzo\n(OSF)', 'Aleatoria\n(RNF)']
failure_sums = df[failure_types].sum()
axes[1].bar(failure_labels, failure_sums.values,
            color=PALETTE_INDUSTRIAL[1:], edgecolor='white', linewidth=1.5, width=0.6)
for i, (val) in enumerate(failure_sums.values):
    axes[1].text(i, val + 1, str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_title('Distribución por Modo de Falla', fontsize=13, fontweight='bold', pad=15)
axes[1].set_ylabel('Número de eventos')

plt.suptitle('Análisis de Fallas — AI4I 2020 Predictive Maintenance Dataset',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_balance_clases.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDesbalance de clases: {failure_counts[0]/failure_counts[1]:.1f}:1')
print('Nota: Dataset severamente desbalanceado. Se aplicará SMOTE para el entrenamiento.')

In [ ]:
# --- 2.2 Distribución de variables continuas por clase ---
continuous_vars = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'
]
var_labels = ['Temp. Aire (K)', 'Temp. Proceso (K)', 'Velocidad (rpm)', 'Torque (Nm)', 'Desgaste (min)']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, (var, label) in enumerate(zip(continuous_vars, var_labels)):
    normal = df[df['Machine failure'] == 0][var]
    falla  = df[df['Machine failure'] == 1][var]
    axes[i].hist(normal, bins=40, alpha=0.6, color=COLOR_OK, label='Normal', density=True)
    axes[i].hist(falla,  bins=40, alpha=0.8, color=COLOR_FAIL, label='Falla', density=True)
    axes[i].set_title(label, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Valor')
    if i == 0:
        axes[i].set_ylabel('Densidad')
        axes[i].legend(fontsize=9)

plt.suptitle('Distribución de Variables de Proceso: Operación Normal vs Falla',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig2_distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 2.3 Matriz de correlación ---
numeric_cols = continuous_vars + ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF']
corr_matrix = df[numeric_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 9}
)
ax.set_title('Matriz de Correlación — Variables de Proceso y Modos de Falla',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('fig3_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

print('Observaciones clave de correlación:')
print('  - Torque y Velocidad: correlación negativa fuerte (relación P = T × ω)')
print('  - Desgaste de herramienta correlaciona con TWF y OSF (esperado)')
print('  - Temperatura de proceso y aire: correlación positiva alta (sistema acoplado)')

In [ ]:
# --- 2.4 Feature Engineering ---
# Creación de variables derivadas con fundamento físico-ingenieril

df_eng = df.copy()

# Delta de temperatura: indicador de eficiencia en disipación de calor
df_eng['Delta_T [K]'] = df_eng['Process temperature [K]'] - df_eng['Air temperature [K]']

# Potencia mecánica: P = Torque × Velocidad angular (ω = rpm × 2π/60)
df_eng['Power [W]'] = df_eng['Torque [Nm]'] * df_eng['Rotational speed [rpm]'] * (2 * np.pi / 60)

# Índice de sobresfuerzo: proxy del criterio OSF del dataset
df_eng['Overstrain_index'] = df_eng['Torque [Nm]'] * df_eng['Tool wear [min]']

# Codificación del tipo de producto
le = LabelEncoder()
df_eng['Type_encoded'] = le.fit_transform(df_eng['Type'])

print('Variables de ingeniería creadas:')
print('  Delta_T [K]        — Diferencial de temperatura proceso-aire')
print('  Power [W]          — Potencia mecánica calculada')
print('  Overstrain_index   — Proxy del índice de sobresfuerzo (Torque × Desgaste)')
print()
print(df_eng[['Delta_T [K]', 'Power [W]', 'Overstrain_index']].describe().round(2))

In [ ]:
# --- 2.5 Análisis por tipo de producto ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Tasa de falla por tipo
failure_by_type = df_eng.groupby('Type')['Machine failure'].agg(['sum', 'count'])
failure_by_type['rate'] = failure_by_type['sum'] / failure_by_type['count'] * 100
failure_by_type['rate'].plot(
    kind='bar', ax=axes[0], color=PALETTE_INDUSTRIAL[1:4],
    edgecolor='white', linewidth=1.5, width=0.5
)
axes[0].set_title('Tasa de Falla por Tipo de Producto', fontsize=11, fontweight='bold')
axes[0].set_ylabel('% Fallas')
axes[0].set_xlabel('Tipo (H=Alta, L=Baja, M=Media calidad)')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(failure_by_type['rate']):
    axes[0].text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=10, fontweight='bold')

# Boxplot de Torque por clase y tipo
df_eng.boxplot(column='Torque [Nm]', by=['Machine failure', 'Type'],
               ax=axes[1], grid=False,
               boxprops=dict(color='#1a3a4a'),
               medianprops=dict(color=COLOR_FAIL, linewidth=2))
axes[1].set_title('Torque por Clase y Tipo de Producto', fontsize=11, fontweight='bold')
axes[1].set_xlabel('(Falla, Tipo)')

# Scatter: Torque vs Velocidad coloreado por Machine failure
for label, color, name in [(0, COLOR_OK, 'Normal'), (1, COLOR_FAIL, 'Falla')]:
    subset = df_eng[df_eng['Machine failure'] == label]
    axes[2].scatter(subset['Rotational speed [rpm]'], subset['Torque [Nm]'],
                    c=color, alpha=0.4, s=8, label=name)
axes[2].set_title('Espacio de Proceso: Velocidad vs Torque', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Velocidad Rotacional (rpm)')
axes[2].set_ylabel('Torque (Nm)')
axes[2].legend()

plt.suptitle('Análisis Segmentado por Tipo de Producto y Variables de Proceso',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_analisis_tipo.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Preparación de Datos para Modelado

### Decisiones de diseño del modelo:

1. **Target:** `Machine failure` (binario) — predicción de falla genérica. Los subtipos (TWF, HDF, PWF, OSF, RNF) se excluyen del modelo para evitar **data leakage** (son causas de la falla, no predictores independientes).
2. **Manejo de desbalance:** Se aplica **SMOTE** (Synthetic Minority Oversampling Technique) únicamente en el conjunto de entrenamiento (Chawla et al., 2002, *Journal of Artificial Intelligence Research*).
3. **Escalado:** StandardScaler aplicado en pipeline para evitar fuga de información entre train/test.
4. **Métrica primaria:** Recall — en mantenimiento, el costo de una falla no detectada (falso negativo) es órdenes de magnitud mayor que el costo de una inspección innecesaria (falso positivo).

In [ ]:
# Definición de features finales
FEATURES = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Type_encoded',
    'Delta_T [K]',
    'Power [W]',
    'Overstrain_index'
]
TARGET = 'Machine failure'

X = df_eng[FEATURES].copy()
y = df_eng[TARGET].copy()

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE solo sobre el conjunto de entrenamiento
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('=== PARTICIÓN DEL DATASET ===')
print(f'Train original:    {X_train.shape[0]:,} registros  |  Fallas: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Train SMOTE:       {X_train_res.shape[0]:,} registros  |  Fallas: {y_train_res.sum()} ({y_train_res.mean()*100:.1f}%)')
print(f'Test (sin SMOTE):  {X_test.shape[0]:,} registros  |  Fallas: {y_test.sum()} ({y_test.mean()*100:.1f}%)')
print()
print('Features seleccionadas:', FEATURES)

---
## 4. Entrenamiento y Evaluación de Modelos

In [ ]:
# Definición de modelos a comparar
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost':             xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                                              scale_pos_weight=y_train.value_counts()[0]/y_train.value_counts()[1],
                                              random_state=42, eval_metric='logloss', verbosity=0)
}

results = {}

for name, model in models.items():
    # XGBoost recibe datos sin SMOTE (maneja desbalance internamente via scale_pos_weight)
    if name == 'XGBoost':
        model.fit(X_train_scaled, y_train)
    else:
        model.fit(X_train_res, y_train_res)
    
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    results[name] = {
        'model':    model,
        'y_pred':   y_pred,
        'y_proba':  y_proba,
        'roc_auc':  roc_auc_score(y_test, y_proba),
        'pr_auc':   average_precision_score(y_test, y_proba),
        'report':   classification_report(y_test, y_pred, output_dict=True)
    }
    
    r = results[name]['report']
    print(f'--- {name} ---')
    print(f"  ROC-AUC: {results[name]['roc_auc']:.4f}   |   PR-AUC: {results[name]['pr_auc']:.4f}")
    print(f"  Recall (falla): {r['1']['recall']:.4f}   |   Precision (falla): {r['1']['precision']:.4f}   |   F1: {r['1']['f1-score']:.4f}")
    print()

In [ ]:
# --- 4.1 Curvas ROC y Precision-Recall ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors_model = ['#2d7d9a', '#52b788', '#e63946']

for (name, res), color in zip(results.items(), colors_model):
    # ROC
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f"{name} (AUC={res['roc_auc']:.3f})")
    # PR
    precision, recall, _ = precision_recall_curve(y_test, res['y_proba'])
    axes[1].plot(recall, precision, color=color, linewidth=2,
                 label=f"{name} (AP={res['pr_auc']:.3f})")

axes[0].plot([0,1], [0,1], 'k--', alpha=0.4, label='Línea base aleatoria')
axes[0].set_xlabel('Tasa de Falsos Positivos', fontsize=11)
axes[0].set_ylabel('Tasa de Verdaderos Positivos (Recall)', fontsize=11)
axes[0].set_title('Curvas ROC — Comparación de Modelos', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Línea base PR (prevalencia de la clase positiva)
baseline_pr = y_test.mean()
axes[1].axhline(y=baseline_pr, color='k', linestyle='--', alpha=0.4,
                label=f'Línea base ({baseline_pr:.3f})')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Curvas Precision-Recall — Métrica Clave para Datos Desbalanceados',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Evaluación de Modelos de Clasificación — Mantenimiento Predictivo',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 4.2 Matrices de Confusión ---
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Falla'],
                yticklabels=['Normal', 'Falla'],
                linewidths=1, linecolor='white')
    ax.set_title(f'{name}\nRecall={res["report"]["1"]["recall"]:.3f} | Precision={res["report"]["1"]["precision"]:.3f}',
                 fontsize=10, fontweight='bold')
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.suptitle('Matrices de Confusión — Conjunto de Prueba',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Interpretabilidad del Modelo — SHAP Values

La interpretabilidad es crítica en entornos industriales: el equipo de mantenimiento necesita entender **por qué** el modelo predice una falla, no solo **que** hay una falla. SHAP (SHapley Additive exPlanations) proporciona la contribución marginal de cada variable a cada predicción individual (Lundberg & Lee, 2017, *NeurIPS*).

In [ ]:
# Usar el modelo con mejor PR-AUC para interpretabilidad
best_model_name = max(results, key=lambda x: results[x]['pr_auc'])
best_model = results[best_model_name]['model']
print(f'Modelo seleccionado para interpretabilidad: {best_model_name}')

# SHAP TreeExplainer (eficiente para modelos de árbol)
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

# Para clasificación binaria, shap_values puede ser lista [clase_0, clase_1]
if isinstance(shap_values, list):
    shap_vals_class1 = shap_values[1]
else:
    shap_vals_class1 = shap_values

# Feature importance global (mean |SHAP|)
fig, ax = plt.subplots(figsize=(10, 6))
feature_names_short = ['T_aire', 'T_proceso', 'Velocidad', 'Torque', 'Desgaste',
                       'Tipo prod.', 'Delta_T', 'Potencia', 'Sobresf. idx']
mean_abs_shap = np.abs(shap_vals_class1).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]

bars = ax.barh(
    [feature_names_short[i] for i in sorted_idx[::-1]],
    mean_abs_shap[sorted_idx[::-1]],
    color=PALETTE_INDUSTRIAL[1], edgecolor='white'
)
ax.set_xlabel('Importancia media |SHAP|', fontsize=11)
ax.set_title(f'Importancia de Variables — {best_model_name}\n(Contribución marginal promedio a la predicción de falla)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. KPIs de Confiabilidad Industrial

Los modelos predictivos son solo una parte del sistema. Los indicadores de confiabilidad permiten **cuantificar el impacto operacional** y comunicar resultados al equipo de gestión en el lenguaje del negocio.

In [ ]:
# --- KPIs de Confiabilidad por Tipo de Producto ---
df_kpi = df_eng.copy()

# Simulación de tiempo entre registros (1 minuto por registro — proceso continuo)
df_kpi['tiempo_acumulado_min'] = df_kpi.index

kpi_results = {}
for tipo in ['H', 'M', 'L']:
    subset = df_kpi[df_kpi['Type'] == tipo]
    n_fallas = subset['Machine failure'].sum()
    n_registros = len(subset)
    
    if n_fallas > 0:
        mtbf = n_registros / n_fallas  # minutos entre fallas
        tasa_falla = n_fallas / n_registros * 100
        disponibilidad = (1 - tasa_falla / 100) * 100
    else:
        mtbf, tasa_falla, disponibilidad = float('inf'), 0, 100
    
    kpi_results[tipo] = {
        'N registros': n_registros,
        'N fallas': int(n_fallas),
        'Tasa falla (%)': round(tasa_falla, 2),
        'MTBF (min)': round(mtbf, 1),
        'MTBF (horas)': round(mtbf / 60, 1),
        'Disponibilidad (%)': round(disponibilidad, 2)
    }

kpi_df = pd.DataFrame(kpi_results).T
print('=== KPIs DE CONFIABILIDAD POR TIPO DE PRODUCTO ===')
print(kpi_df.to_string())

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

tipos = list(kpi_results.keys())
mtbf_vals = [kpi_results[t]['MTBF (horas)'] for t in tipos]
disp_vals  = [kpi_results[t]['Disponibilidad (%)'] for t in tipos]

axes[0].bar(tipos, mtbf_vals, color=PALETTE_INDUSTRIAL[1:4], width=0.4, edgecolor='white')
for i, v in enumerate(mtbf_vals):
    axes[0].text(i, v + 0.5, f'{v:.1f} h', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('MTBF por Tipo de Producto\n(Mean Time Between Failures)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Horas')
axes[0].set_xlabel('Tipo de Producto')

axes[1].bar(tipos, disp_vals, color=PALETTE_INDUSTRIAL[1:4], width=0.4, edgecolor='white')
axes[1].axhline(y=95, color=COLOR_FAIL, linestyle='--', linewidth=1.5,
                label='Benchmark 95% (referencia industria)')
for i, v in enumerate(disp_vals):
    axes[1].text(i, v - 0.3, f'{v:.2f}%', ha='center', va='top', fontsize=11, fontweight='bold',
                 color='white')
axes[1].set_title('Disponibilidad Operacional por Tipo de Producto', fontsize=11, fontweight='bold')
axes[1].set_ylabel('%')
axes[1].set_ylim(90, 101)
axes[1].legend()

plt.suptitle('KPIs de Confiabilidad Industrial — Dashboard TPM',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig8_kpis_confiabilidad.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Exportación para Power BI

Los siguientes archivos CSV alimentarán el dashboard de Power BI con las tablas necesarias para el modelo estrella de datos.

In [ ]:
# --- Tabla de hechos principal con predicciones ---
best_res = results[best_model_name]

# Agregar predicciones al dataset completo
X_all_scaled = scaler.transform(df_eng[FEATURES])
df_eng['pred_failure'] = best_model.predict(X_all_scaled)
df_eng['pred_proba']   = best_model.predict_proba(X_all_scaled)[:, 1]
df_eng['pred_correcto'] = (df_eng['pred_failure'] == df_eng['Machine failure']).astype(int)

# Tabla de hechos: datos de proceso con predicción
fact_table = df_eng[[
    'UDI', 'Product ID', 'Type',
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
    'Delta_T [K]', 'Power [W]', 'Overstrain_index',
    'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF',
    'pred_failure', 'pred_proba'
]].copy()

# Tabla de KPIs
kpi_export = kpi_df.reset_index().rename(columns={'index': 'Product_Type'})

# Tabla de resumen de modelos
model_summary = pd.DataFrame([
    {
        'Modelo': name,
        'ROC_AUC': round(res['roc_auc'], 4),
        'PR_AUC': round(res['pr_auc'], 4),
        'Recall_falla': round(res['report']['1']['recall'], 4),
        'Precision_falla': round(res['report']['1']['precision'], 4),
        'F1_falla': round(res['report']['1']['f1-score'], 4)
    }
    for name, res in results.items()
])

# Exportar
fact_table.to_csv('pm_fact_table.csv', index=False)
kpi_export.to_csv('pm_kpis_confiabilidad.csv', index=False)
model_summary.to_csv('pm_model_comparison.csv', index=False)

print('Archivos exportados para Power BI:')
print('  pm_fact_table.csv          — Tabla de hechos principal (10,000 registros)')
print('  pm_kpis_confiabilidad.csv  — KPIs por tipo de producto')
print('  pm_model_comparison.csv    — Comparación de modelos')
print()
print(model_summary.to_string(index=False))

---
## 8. Conclusiones y Recomendaciones

### Hallazgos técnicos

1. **El dataset presenta un desbalance severo de clases** (96.6% normal / 3.4% falla), típico de operaciones industriales bien mantenidas. Este desbalance exige métricas de evaluación adecuadas (PR-AUC) y técnicas de re-muestreo (SMOTE) para evitar un modelo que prediga siempre "sin falla" con alta accuracy pero nula utilidad operacional.

2. **Las variables de mayor importancia predictiva** son: Torque, Desgaste de herramienta, el índice de sobresfuerzo (variable derivada) y la Potencia mecánica calculada. Esto es consistente con los mecanismos físicos de los modos de falla documentados en el dataset (Matzka, 2020).

3. **El mejor modelo** logra un Recall sobre fallas significativamente superior a la línea base aleatoria, lo que en términos operacionales se traduce en detección anticipada de una fracción mayor de fallas reales con tasas de falsas alarmas manejables.

4. **El MTBF varía por tipo de producto**, lo que sugiere que los planes de mantenimiento preventivo deben diferenciarse por categoría de producción, no aplicarse de forma homogénea.

### Recomendaciones operacionales

- Establecer **umbrales de alerta diferenciados** por tipo de producto basados en la probabilidad predicha, no en el umbral por defecto de 0.5 (ajustar según el trade-off Recall/Precision específico de cada operación)
- Integrar las variables derivadas (Power, Overstrain_index) en el sistema MES como KPIs de proceso en tiempo real
- Implementar monitoreo continuo del delta de temperatura (Delta_T) como indicador temprano de deterioro en la disipación de calor
- Reentrenar el modelo periódicamente a medida que se acumulen datos históricos reales de la planta

### Referencias

- Matzka, S. (2020). *Explainable Artificial Intelligence for Predictive Maintenance Applications*. UCI ML Repository. https://doi.org/10.24432/C5HS5C
- Chawla, N.V. et al. (2002). SMOTE: Synthetic Minority Over-sampling Technique. *Journal of Artificial Intelligence Research*, 16, 321-357.
- Lundberg, S.M. & Lee, S.I. (2017). A Unified Approach to Interpreting Model Predictions. *NeurIPS*.
- Mobley, R.K. (2002). *An Introduction to Predictive Maintenance* (2nd ed.). Butterworth-Heinemann.
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3rd ed.). O'Reilly.